# 01 — Recent 6 Months

In [2]:
import pandas as pd
from datetime import datetime, timedelta
from nba_api.stats.endpoints import leaguegamelog
from pathlib import Path


In [3]:
def season_str_for_date(d: datetime) -> str:
    """
    Return NBA season string like '2025-26' for a given date.
    NBA seasons start in Oct (use Aug as switch to be safe for preseason spillover).
    """
    y = d.year
    if d.month >= 8:  # Aug+ → new season start
        return f"{y}-{(y + 1) % 100:02d}"
    else:
        return f"{y - 1}-{y % 100:02d}"


In [4]:
def fetch_last_six_months_regular_season():
    """
    Pull the last ~6 months of NBA TEAM game logs from the official NBA Stats API,
    Regular Season only. If the 6-month window crosses a season boundary,
    query both seasons and union results.
    """
    today = datetime.today()
    start = today - timedelta(days=182)  # ≈ 6 months

    # API requires MM/DD/YYYY
    date_from = start.strftime("%m/%d/%Y")
    date_to   = today.strftime("%m/%d/%Y")

    # Determine which seasons to query (may be one or two)
    seasons = sorted({season_str_for_date(start), season_str_for_date(today)})

    frames = []
    for s in seasons:
        print(f"Querying season {s} | {date_from} → {date_to}")
        logs = leaguegamelog.LeagueGameLog(
            season=s,
            season_type_all_star='Regular Season',
            date_from_nullable=date_from,
            date_to_nullable=date_to,
            player_or_team_abbreviation='T',  # team logs
            sorter='DATE',
            direction='ASC'
        )
        df_s = logs.get_data_frames()[0]
        print(f"  rows returned: {len(df_s)}")
        frames.append(df_s)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)

    # Normalize column names now for filtering
    df.columns = [c.lower() for c in df.columns]

    # Ensure datetime and clamp strictly to [start, today]
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
        df = df[(df['game_date'] >= start) & (df['game_date'] <= today)]

    # De-duplicate (one row per team per game)
    dedup_cols = [c for c in ['game_id', 'team_id'] if c in df.columns]
    if dedup_cols:
        before = len(df)
        df = df.drop_duplicates(subset=dedup_cols, keep='last')
        print(f"De-duplicated: {before} → {len(df)}")

    return df.reset_index(drop=True)


In [5]:
df = fetch_last_six_months_regular_season()
print("Total rows fetched (team-rows):", len(df))
df.head(3)

Querying season 2025-26 | 09/11/2025 → 03/12/2026
  rows returned: 1960
De-duplicated: 1960 → 1960
Total rows fetched (team-rows): 1960


,season_id,team_id,team_abbreviation,team_name,game_id,game_date,matchup,wl,min,fgm,...,dreb,reb,ast,stl,blk,tov,pf,pts,plus_minus,video_available
0,22025,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,240,38,...,31,40,29,10,4,19,27,119,10,1
1,22025,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,240,42,...,32,39,23,7,2,20,21,109,-10,1
2,22025,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,290,43,...,36,52,23,6,5,25,26,124,-1,1


In [6]:
# Save location for the cleaned dataset used by feature engineering
OUT_PATH = Path("../data/clean/clean_game_nba.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Lower-case columns (if not already)
df.columns = [c.lower() for c in df.columns]

# Force datetime type again (safe)
if 'game_date' in df.columns:
    df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')

df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned 6‑month dataset → {OUT_PATH}")
print(f"Games saved: {len(df)}")

Saved cleaned 6‑month dataset → ..\data\clean\clean_game_nba.csv
Games saved: 1960


In [7]:
if not df.empty:
    last_date = df['game_date'].max()
    first_date = df['game_date'].min()
    teams = sorted(set(df.get('team_name', pd.Series(dtype=str)).dropna()))
    print(f"Date range: {first_date.date()} → {last_date.date()}")
    print(f"Unique teams (by label): {len(teams)}")
    print(teams[:10], "...")
else:
    print("No rows pulled — verify season/date window and try again.")


Date range: 2025-10-21 → 2026-03-11
Unique teams (by label): 30
['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Hornets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Detroit Pistons', 'Golden State Warriors'] ...
